In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.isdir("smartstock-ai"):          # Colab/Kaggle: clone your repo first
    subprocess.run(["git", "clone", "https://github.com/<you>/smartstock-ai.git"], check=True); os.chdir("smartstock-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "backend")) and ROOT != "/": ROOT = os.path.dirname(ROOT)
os.chdir(ROOT); sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "training", "scripts")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RAW_OK = os.path.exists("data/raw/m5/sales_train_validation.csv")
if not RAW_OK:   # no real M5 download present -> generate M5-format synthetic data so the notebook runs anywhere
    subprocess.run([sys.executable, "training/scripts/make_synthetic_m5.py"], check=True)
if not os.path.exists("data/processed/panel.npz"):
    subprocess.run([sys.executable, "training/scripts/prepare_data.py"], check=True)
def ensure_trained():
    if not os.path.exists("training/outputs/metrics.json"):
        subprocess.run([sys.executable, "training/scripts/train.py"], check=True)

# 03 · Data preparation and leakage-safe features
Wide→long, joins and QA live in `training/scripts/prepare_data.py`. Features come from `backend/app/features/feature_builder.py` — the *same* code the API runs.

In [ ]:
d = pd.read_parquet("data/processed/sales_daily.parquet"); print(d.shape); display(d.head())
from app.features.feature_builder import make_features, FEATURES, CAL_KEYS
z = np.load("data/processed/panel.npz"); import json; meta = json.load(open("data/processed/series.json"))["series"]
cal = {k: z[f"cal_{k}"] for k in CAL_KEYS}; idx = {k: np.array([s[k] for s in meta]) for k in ["item_idx","store_idx","dept_idx","cat_idx"]}
X = make_features(z["sales"], z["price"], cal, idx, [600], with_target=True); print(len(FEATURES), "features;", X.shape); display(X[FEATURES[:10] + ["y"]].head())

In [ ]:
# Leakage check: corrupt every observation AFTER the origin -> features must not change
s2 = z["sales"].copy(); s2[:, 601:] = 9999
X2 = make_features(s2, z["price"], cal, idx, [600], with_target=False)
assert np.array_equal(X[FEATURES].values, X2[FEATURES].values); print("no future information in features ✔")